In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))   # notebooks/ -> src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from models import get_models
from evaluation import BORDERLINE, evaluate, evaluate_lstm_ae, results_table

In [ ]:
# reading the data 
df = pd.read_csv("input-data/Project 2 - BRPMM/process A/calibration data/calibration_data.csv")
df.head()

In [ ]:
# checking for missing values with count
df.describe().T

In [ ]:
# reading the labels 
df_labels = pd.read_csv("input-data/Project 2 - BRPMM/process A/calibration data/calibration_labels.csv")
df_labels.head()

In [ ]:
# checking for missing values in the labels
df_labels.describe().T

In [ ]:
# seeing whether the data is time series 
import matplotlib.pyplot as plt
import seaborn as sns

df["batch_id"].value_counts().plot.bar(); plt.show()
df.groupby("batch_id").size().plot.hist(bins=30); plt.title("rows per batch"); plt.show()

In [ ]:
# merging the data and the labels
df = df.merge(df_labels, on="batch_id", how="left")
df["label"].isna().sum()   # rows that didn't get a label: should be 0, or explainable

In [ ]:
# inspecting the new merged data
df.head()

In [ ]:
df["t"] = df.groupby("batch_id").cumcount()   # 0, 1, 2, ... restarts for each batch

In [ ]:
df.head()

In [ ]:
# counting the time series length of each batch
lengths = df.groupby("batch_id").size()
lengths.describe()
lengths.value_counts().head()

In [ ]:
# creating a dataframe with one row per batch, making features, creating "borderline" batches
import numpy as np

features = ["flow_rate_1", "density", "flow_rate_2", "pressure_1", "temperature_1",
            "temperature_2", "temperature_3", "temperature_4",
            "pressure_2", "pressure_3", "pressure_4"]

batch_feats = df.groupby("batch_id")[features].agg(["mean", "std", "min", "max"])
batch_feats.columns = ["_".join(c) for c in batch_feats.columns]

t_c = df["t"] - df.groupby("batch_id")["t"].transform("mean")
x_c = df[features] - df.groupby("batch_id")[features].transform("mean")
slopes = x_c.mul(t_c, axis=0).groupby(df["batch_id"]).sum().div(
    (t_c ** 2).groupby(df["batch_id"]).sum(), axis=0)
slopes.columns = [f"{f}_slope" for f in features]

batch_feats = batch_feats.join(slopes).join(df_labels.set_index("batch_id"))

# mark the borderline batches from the brief
batch_feats["group"] = batch_feats["label"]
batch_feats.loc[batch_feats.index.isin(["A_101", "A_102", "A_103", "A_104"]), "group"] = "borderline"

print(batch_feats.shape)
batch_feats["group"].value_counts()

In [ ]:
# inspecting the time series, plotting. good batches are in blue, bad are in red
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 3, figsize=(15, 12))
for ax, f in zip(axes.flat, features):
    for b, g in df.groupby("batch_id"):
        rej = g["label"].iloc[0] == "rejected"
        ax.plot(g["t"][::10], g[f][::10], color="red" if rej else "blue",
                alpha=0.5 if rej else 0.15, lw=0.7)
    ax.set_title(f)
plt.tight_layout(); plt.show()

In [ ]:
# creating a PCA map of all batches 
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import seaborn as sns

feat_cols = [c for c in batch_feats.columns if c not in ["label", "group"]]
X = batch_feats[feat_cols]
acc = batch_feats["label"] == "accepted"

scaler = StandardScaler().fit(X[acc])      # fit on accepted only
pca = PCA(n_components=2).fit(scaler.transform(X[acc]))
Z = pca.transform(scaler.transform(X))

sns.scatterplot(x=Z[:, 0], y=Z[:, 1], hue=batch_feats["group"], s=50)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%})")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%})")
plt.title("Each dot = one batch"); plt.show()

In [ ]:
# leave-one-batch-out anomaly detection (see evaluation.evaluate)
# The model only ever learns from accepted batches.
# Each accepted batch gets scored by a model that never saw it.
# Rejected batches are scored by every fold's model and averaged.
from sklearn.metrics import roc_auc_score

acc_scores, rej_scores = evaluate(get_models()["Isolation Forest"], X, acc)

scores = pd.concat([acc_scores, rej_scores])       # higher = more anomalous
y = (batch_feats.loc[scores.index, "label"] == "rejected").astype(int)
print("ROC AUC:", round(roc_auc_score(y, scores), 3))

In [ ]:
# picking a threshold and counting the mistakes
threshold = acc_scores.quantile(0.70)   # allows ~30% false alarms on good batches (0.95 -> ~5%)

pred = (scores > threshold).astype(int)
tp = ((pred == 1) & (y == 1)).sum(); fn = ((pred == 0) & (y == 1)).sum()
fp = ((pred == 1) & (y == 0)).sum(); tn = ((pred == 0) & (y == 0)).sum()
print(f"Caught {tp} of {tp+fn} rejected batches, missed {fn}")
print(f"False alarms: {fp} of {fp+tn} accepted batches")

plt.hist(acc_scores, bins=30, alpha=0.6, label="accepted")
plt.hist(rej_scores, bins=30, alpha=0.6, label="rejected", color="red")
plt.axvline(threshold, color="k", ls="--", label="threshold")
plt.xlabel("anomaly score"); plt.legend(); plt.show()

print(scores[scores.index.isin(BORDERLINE)])  # where do the borderline ones land?

In [ ]:
# compare all tabular detectors (models.get_models) with the same leave-one-batch-out scheme
import warnings; warnings.filterwarnings("ignore")

all_scores = {}
for name, model in get_models().items():
    print("running", name, "...")
    all_scores[name] = evaluate(model, X, acc)

results_df = results_table(all_scores)
results_df.style.format(precision=3)

In [ ]:
from sklearn.metrics import roc_curve

for name, (acc_s, rej_s) in all_scores.items():
    scores = pd.concat([acc_s, rej_s])
    y = scores.index.isin(rej_s.index).astype(int)
    fpr, tpr, _ = roc_curve(y, scores)
    plt.plot(fpr, tpr, label=f"{name} ({results_df.loc[name, 'ROC AUC']:.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8)
plt.xlabel("false alarm rate"); plt.ylabel("detection rate"); plt.legend(fontsize=8)
plt.title("Leave-one-batch-out ROC, process A"); plt.show()

In [ ]:
# LSTM autoencoder on the raw time series (resampled to 100 steps), 5-fold over accepted batches
all_scores["LSTM autoencoder"] = evaluate_lstm_ae(df, batch_feats["label"], features)

results_df = results_table(all_scores)
results_df.style.format(precision=3)